# 04 · GloVe: vetores globais a partir de razões de probabilidades

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


**Artigo-base (leitura complementar):** Pennington, Socher & Manning (2014), *GloVe: Global Vectors for Word Representation* (EMNLP).

## Ideia central
O word2vec percorre o corpus janela a janela e **nunca usa as estatísticas globais** explicitamente; os métodos de contagem (LSA/HAL, notebook 01) usam a matriz global mas ponderam mal (todas as coocorrências pesam igual, e as frequentes dominam). O GloVe propõe treinar diretamente sobre a matriz de coocorrência $X$, com um objetivo de mínimos quadrados **ponderado**:

$$J = \sum_{i,j=1}^{V} f(X_{ij})\,\big(w_i^{\top}\tilde w_j + b_i + \tilde b_j - \log X_{ij}\big)^2,\qquad f(x) = \begin{cases} (x/x_{\max})^{\alpha} & x < x_{\max} \\ 1 & \text{caso contrário}\end{cases}$$

com $x_{\max} = 100$ e $\alpha = 3/4$. O argumento do artigo: o que carrega significado não é $P(k\mid i)$, mas a **razão** $P(k\mid i)/P(k\mid j)$ — e um modelo log-bilinear $w_i^\top \tilde w_k = \log P(k\mid i)$ é o jeito mais simples de capturar razões como diferenças de vetores.

Em Hartmann et al. (2017) o GloVe foi o **melhor nas analogias** (sobretudo semânticas) e o **pior nas tarefas extrínsecas** — o paradoxo que discutiremos no notebook 07.

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: /home/user/linguagem-blis2s26 | fast: False


## 1. A motivação: razões de probabilidade (Tabela 1 do artigo)

O artigo usa *ice*/*steam*: para $k$ = *solid*, $P(k|ice)/P(k|steam) \gg 1$; para *gas*, $\ll 1$; para *water* e *fashion*, $\approx 1$. Repetimos a ideia em português no corpus Machado com os alvos **rei**/**padre** — dois substantivos masculinos de autoridade com contextos distintos.

In [2]:
from replang.data.corpora import load_machado_sentences
from replang.models.cooccurrence import build_vocab, cooccurrence_matrix
from replang.models.glove_np import glove_probability_ratios
sents = load_machado_sentences()
words, index, freq = build_vocab(sents, min_count=5)
X = cooccurrence_matrix(sents, index, window=5, weighting="harmonic")
probes = ["trono", "coroa", "igreja", "missa", "casa", "dia", "mulher", "dinheiro"]
glove_probability_ratios(X, words, "rei", "padre", probes).round(4)

,P(k|rei),P(k|padre),P(k|rei)/P(k|padre)
trono,0.0021,0.0000,NaN
coroa,0.0008,0.0000,NaN
igreja,0.0000,0.0002,0.0000
missa,0.0000,0.0008,0.0000
casa,0.0003,0.0025,0.1021
dia,0.0007,0.0003,2.2605
mulher,0.0005,0.0006,0.8616
dinheiro,0.0000,0.0000,NaN


Leia a última coluna: *trono* e *coroa* ≫ 1 (associadas a *rei*), *igreja* e *missa* ≪ 1 (associadas a *padre*), *casa*, *dia*, *dinheiro* ≈ 1 (neutras). A razão **cancela** o que é comum aos dois e realça o que discrimina — é isso que o produto interno $w_i^\top \tilde w_k$ deve modelar.

## 2. A função de ponderação $f(x)$

Três exigências do artigo: $f(0)=0$ (pares que nunca coocorrem não entram — a soma só percorre os $X_{ij}>0$, por isso é esparsa); $f$ não decrescente (pares raros pesam menos — são ruidosos); $f$ **saturada** (pares frequentíssimos não dominam — o problema do LSA).

In [3]:
import plotly.express as px
x = np.linspace(0, 150, 300)
df = pd.DataFrame({"x": x, "α=3/4, x_max=100": np.where(x < 100, (x / 100) ** 0.75, 1), "α=1 (linear)": np.where(x < 100, x / 100, 1), "x_max=50": np.where(x < 50, (x / 50) ** 0.75, 1)})
px.line(df.melt(id_vars="x"), x="x", y="value", color="variable", template="plotly_white", title="Função de ponderação f(X_ij) do GloVe").show()
nz = X.data
print(f"pares não nulos: {X.nnz:,} de {len(words)**2:,} possíveis ({X.nnz/len(words)**2:.2%}); mediana de X_ij = {np.median(nz):.2f}; 99º percentil = {np.percentile(nz, 99):.1f}")

pares não nulos: 4,505,956 de 537,961,636 possíveis (0.84%); mediana de X_ij = 0.50; 99º percentil = 19.9


## 3. Implementação *from scratch* (numpy + AdaGrad)

`replang.models.glove_np.GloVe` constrói $X$ com ponderação $1/d$ (distância na janela, como no artigo), inicializa $W, \tilde W, b, \tilde b$ e minimiza $J$ por SGD com **AdaGrad** (o otimizador do artigo) sobre lotes de pares $(i, j)$ não nulos. No final, o vetor da palavra é $W + \tilde W$ (§4.2 do artigo: somar os dois reduz ruído).

In [4]:
import inspect
from replang.models.glove_np import GloVe
src = inspect.getsource(GloVe.fit)
print(src[src.index("for epoch"):src.index("rec = {")])

for epoch in range(self.epochs):
            order = self.rng.permutation(n)
            total = 0.0
            for b0 in range(0, n, self.batch_size):
                sl = order[b0 : b0 + self.batch_size]
                ii, jj = i[sl], j[sl]
                wi, wj = self.W[ii], self.Wc[jj]
                diff = (wi * wj).sum(1) + self.b[ii] + self.bc[jj] - logx[sl]
                wdiff = fx[sl] * diff
                total += float((wdiff * diff).sum())
                g = 2 * wdiff  # dJ/d(pred)
                gw_i = g[:, None] * wj
                gw_j = g[:, None] * wi
                # AdaGrad: atualização ∝ g / sqrt(acumulado)
                np.add.at(self.W, ii, -self.lr * gw_i / np.sqrt(gW[ii]))
                np.add.at(self.Wc, jj, -self.lr * gw_j / np.sqrt(gWc[jj]))
                np.add.at(self.b, ii, -self.lr * g / np.sqrt(gb[ii]))
                np.add.at(self.bc, jj, -self.lr * g / np.sqrt(gbc[jj]))
                np.add.at(gW, ii, gw_i**2)
                np.ad

In [5]:
import time
from replang.data.corpora import toy_corpus
toy = toy_corpus() * 20
t = time.time()
g = GloVe(dim=20, window=2, epochs=60, x_max=10, lr=0.05, seed=1).fit(toy)
wv_g = g.to_wordvectors("GloVe toy")
print(f"{time.time()-t:.1f}s | perda final {g.history[-1]['loss']:.4f}")
from replang.viz import training_curve
training_curve(g.history, title="GloVe (toy): J ponderado por época").show()
for w in ["rei", "gato", "paris"]:
    print(f"{w:<6} {[x for x, _ in wv_g.most_similar(w, topn=4)]}")

0.0s | perda final 0.1136


rei    ['menino', 'cachorro', 'homem', 'da']
gato   ['late', 'peixe', 'um', 'menina']
paris  ['roma', 'governa', 'cozinha', 'reino']


## 4. GloVe no corpus Machado e comparação com Skip-gram e PPMI-SVD

O modelo `machado_glove100` (numpy, 15 épocas, janela 5) foi treinado por `replang train`. Comparamos com o Skip-gram (gensim) e com o pipeline de contagem do notebook 01 em três frentes: vizinhos, analogias PT-BR (sintáticas) e a **mini-similaridade** de palavras (`MINI_WORDSIM_PT`).

In [6]:
from replang.data.embeddings import load_local_model
from replang.models.cooccurrence import CountModel
from replang.data.analogies import load_analogies
from replang.eval.analogies import evaluate_analogies
from replang.eval.similarity import word_similarity_eval
glove = load_local_model("machado_glove100"); sg = load_local_model("machado_sg100")
cm = CountModel(window=5, dim=100, min_count=5, alpha=0.75, weighting="harmonic", max_vocab=25000).fit(sents)
ppmi_svd = cm.to_wordvectors("PPMI-SVD 100d (Machado)")
an = load_analogies("pt-br")
rows = []
for wv in [glove, sg, ppmi_svd]:
    r = evaluate_analogies(wv, an, restrict=None).set_index("categoria")
    ws = word_similarity_eval(wv)
    rows.append({"modelo": wv.name, "analogias sintáticas": r.loc["TOTAL sintática", "acurácia"], "analogias semânticas": r.loc["TOTAL semântica", "acurácia"],
                 "cobertas": int(r.loc["TOTAL", "cobertas"]), "mini-WordSim ρ": ws["spearman"]})
pd.DataFrame(rows).style.format({"analogias sintáticas": "{:.1%}", "analogias semânticas": "{:.1%}", "mini-WordSim ρ": "{:.2f}"})

,modelo,analogias sintáticas,analogias semânticas,cobertas,mini-WordSim ρ
0,"GloVe 100d (numpy, Machado)",0.5%,12.4%,3771,-0.31
1,machado_sg100,3.1%,19.5%,3771,-0.02
2,PPMI-SVD 100d (Machado),5.5%,28.8%,3771,-0.13


In [7]:
for w in ["capitu", "amor", "dinheiro", "rua", "ontem"]:
    print(f"{w:<9} GloVe:    {[x for x, _ in glove.most_similar(w, topn=6)]}")
    print(f"{'':<9} SG:       {[x for x, _ in sg.most_similar(w, topn=6)]}")
    print(f"{'':<9} PPMI-SVD: {[x for x, _ in ppmi_svd.most_similar(w, topn=6)]}")

capitu    GloVe:    ['cecília', 'sofia', 'ângela', 'guiomar', 'casar', 'helena']
          SG:       ['lalau', 'ângela', 'augusta', 'mariana', 'amélia', 'espantada']
          PPMI-SVD: ['virgília', 'escobar', 'sofia', 'flora', 'carmo', 'natividade']
amor      GloVe:    ['nome', 'coração', 'costume', 'sentimento', 'poeta', 'respeito']
          SG:       ['amante', 'afeto', 'gozo', 'orgulho', 'vínculo', 'fruto']
          PPMI-SVD: ['coração', 'sentimento', 'afeto', 'espírito', 'orgulho', 'pensamento']
dinheiro  GloVe:    ['perder', 'cuidado', 'tempo', 'senão', 'algum', 'teria']
          SG:       ['pagar', 'preço', 'emprego', 'ganho', 'ganhar', 'gastar']
          PPMI-SVD: ['preço', 'pagar', 'emprego', 'lhes', 'remédio', 'fortuna']
rua       GloVe:    ['ouvidor', 'véspera', 'igreja', 'cama', 'abaixo', 'mesa']
          SG:       ['esquina', 'quitanda', 'ouvidor', 'ourives', 'loja', 'ajuda']
          PPMI-SVD: ['esquina', 'ouvidor', 'quitanda', 'direção', 'porta', 'praia']
ontem    

## 5. Contagem × predição: o debate

* Baroni, Dinu & Kruszewski (2014), *Don't count, predict!*: modelos preditivos (word2vec) venceram os de contagem em quase todos os testes.
* Pennington et al. (2014): GloVe, um modelo de contagem bem ponderado, iguala ou supera o word2vec em analogias com menos tempo de treino (Fig. 4 do artigo: GloVe converge em menos iterações/tempo que CBOW/Skip-gram).
* Levy, Goldberg & Dagan (2015): a diferença são **hiperparâmetros** (janela dinâmica, subamostragem, suavização $\alpha$, deslocamento, vetor de contexto somado, normalização). Transplantados para PPMI-SVD, as diferenças quase somem.
* Hartmann et al. (2017), em PT: GloVe melhor em analogias **semânticas** (48,5 % com 600d), mas pior em POS e similaridade de sentenças.

A lição prática: *escolha pela tarefa e ajuste os hiperparâmetros*, não pela "família" do algoritmo.

In [8]:
# Curva de treino do GloVe 100d (Machado) gravada pelo script de treino
import json
meta = json.loads((PATHS.models / "machado_glove100.json").read_text())
training_curve(meta["history"], title=f"GloVe 100d (Machado): perda por época — {meta['seconds']}s de treino").show()

## 6. Síntese e pontos para a apresentação

1. **Razões de probabilidade** como objeto de modelagem: o insight do artigo, independente do algoritmo.
2. O objetivo é um **mínimos quadrados ponderado** sobre $\log X_{ij}$: explícito, esparso, paralelizável; AdaGrad.
3. $W + \tilde W$: GloVe soma os dois conjuntos de vetores; word2vec descarta $W_{out}$.
4. Em PT (Hartmann), GloVe ≠ "melhor" — depende da tarefa.

### Perguntas para discussão
* Por que $\log X_{ij}$ e não $X_{ij}$? (lei de Zipf; o log torna a relação aproximadamente linear)
* O que acontece com $f$ se $x_{\max}$ for muito baixo? (todos os pares pesam igual → volta ao LSA)

## Referências
- PENNINGTON, J.; SOCHER, R.; MANNING, C. **GloVe: Global Vectors for Word Representation.** EMNLP, 2014. [ACL D14-1162](https://aclanthology.org/D14-1162/)
- BARONI, M.; DINU, G.; KRUSZEWSKI, G. **Don't count, predict!** ACL, 2014.
- LEVY, O.; GOLDBERG, Y. **Neural Word Embedding as Implicit Matrix Factorization.** NIPS, 2014; LEVY, GOLDBERG & DAGAN. **Improving Distributional Similarity with Lessons Learned from Word Embeddings.** TACL, 2015.
- HARTMANN, N. et al. **Portuguese Word Embeddings: Evaluating on Word Analogies and Natural Language Tasks.** STIL, 2017. [arXiv:1708.06025](https://arxiv.org/abs/1708.06025)